# 02 - ACE Training
Train the localized ACE model on the dataset and log metrics.


In [1]:
import sys
import torch
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch_geometric.loader import DataLoader
import pandas as pd

torch.manual_seed(42)

sys.path.append("..")
from src.dataset import MDTrajectoryDataset
from src.models.ace_wrapper import ACEWrapper
from src.trainer import BenchmarkTrainer

In [2]:
# Load static datasets ONCE before the loop to save time
print('Pre-loading static datasets...')
val_ds = MDTrajectoryDataset('../data/val.extxyz', cutoff=5.0)
test_ds = MDTrajectoryDataset('../data/test.extxyz', cutoff=5.0)

import time

fractions = [10, 40, 70, 100]
metrics_dict = {}

for frac in fractions:
    print(f"\n{'='*40}")
    print(f"Starting ACE Training for {frac}% data fraction")
    print(f"{'='*40}\n")
    
    # 1. Load Data
    train_path = f"../data/train_{frac}.extxyz"
    train_ds = MDTrajectoryDataset(train_path, cutoff=5.0)
    
    train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, pin_memory=True)

    # 2. Re-Initialize ACE model (resets weights)
    model = ACEWrapper(
        num_elements=120,
        num_radial=8,
        l_max=2,
        r_cut=5.0,
        hidden_dim=32
    )

    optimizer = AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

    trainer = BenchmarkTrainer(
        model=model,
        optimizer=optimizer,
        scheduler=scheduler,
        train_loader=train_loader,
        val_loader=val_loader,
        device="cuda" if torch.cuda.is_available() else "cpu",
        energy_weight=1.0,
        force_weight=100.0
    )
    
    # 3. Train
    metrics_df = trainer.train(max_epochs=30, patience=10)
    metrics_df.to_csv(f"../data/ace_metrics_{frac}.csv", index=False)
    
    # 4. Test (In-Distribution)
    test_metrics = trainer.test_epoch(test_loader)
    pd.DataFrame([test_metrics]).to_csv(f"../data/ace_test_metrics_{frac}.csv", index=False)
    
    
    # 6. Save model
    torch.save(model.state_dict(), f"../data/ace_model_{frac}.pth")
    
    metrics_dict[frac] = metrics_df
    
    # 7. Free GPU Memory
    del model, optimizer, trainer, train_loader, val_loader, test_loader
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


Pre-loading static datasets...
Pre-computing graphs for 200 structures (cutoff=5.0 Å)...


Building graphs:   0%|          | 0/200 [00:00<?, ?it/s]

  Done. Dataset ready (200 graphs).
Pre-computing graphs for 200 structures (cutoff=5.0 Å)...


Building graphs:   0%|          | 0/200 [00:00<?, ?it/s]

  Done. Dataset ready (200 graphs).

Starting ACE Training for 10% data fraction

Pre-computing graphs for 100 structures (cutoff=5.0 Å)...


...

  Done. Dataset ready (400 graphs).
Epoch 000 | Time: 0.61s | Train E MAE: 92.20 meV/atom | Train F MAE: 203.63 meV/Å | Val E MAE: 61.12 meV/atom | Val F MAE: 198.32 meV/Å
Epoch 001 | Time: 0.74s | Train E MAE: 35.08 meV/atom | Train F MAE: 198.97 meV/Å | Val E MAE: 28.49 meV/atom | Val F MAE: 192.47 meV/Å
Epoch 002 | Time: 0.40s | Train E MAE: 22.50 meV/atom | Train F MAE: 192.18 meV/Å | Val E MAE: 9.05 meV/atom | Val F MAE: 183.25 meV/Å
Epoch 003 | Time: 0.38s | Train E MAE: 12.25 meV/atom | Train F MAE: 177.93 meV/Å | Val E MAE: 13.18 meV/atom | Val F MAE: 162.65 meV/Å
Epoch 004 | Time: 0.39s | Train E MAE: 7.73 meV/atom | Train F MAE: 148.46 meV/Å | Val E MAE: 3.51 meV/atom | Val F MAE: 121.33 meV/Å
Epoch 005 | Time: 0.38s | Train E MAE: 5.53 meV/atom | Train F MAE: 97.71 meV/Å | Val E MAE: 6.86 meV/atom | Val F MAE: 73.36 meV/Å
Epoch 006 | Time: 0.38s | Train E MAE: 4.62 meV/atom | Train F MAE: 71.38 meV/Å | Val E MAE: 4.52 meV/atom | Val F MAE: 63.74 meV/Å
Epoch 007 | Time: 0.38s

Building graphs:   0%|          | 0/700 [00:00<?, ?it/s]

  Done. Dataset ready (700 graphs).
Epoch 000 | Time: 0.81s | Train E MAE: 42.07 meV/atom | Train F MAE: 204.56 meV/Å | Val E MAE: 33.57 meV/atom | Val F MAE: 196.27 meV/Å
Epoch 001 | Time: 0.65s | Train E MAE: 15.59 meV/atom | Train F MAE: 189.15 meV/Å | Val E MAE: 15.68 meV/atom | Val F MAE: 169.26 meV/Å
Epoch 002 | Time: 0.68s | Train E MAE: 8.17 meV/atom | Train F MAE: 132.37 meV/Å | Val E MAE: 2.64 meV/atom | Val F MAE: 74.30 meV/Å
Epoch 003 | Time: 0.64s | Train E MAE: 5.58 meV/atom | Train F MAE: 60.08 meV/Å | Val E MAE: 1.47 meV/atom | Val F MAE: 48.02 meV/Å
Epoch 004 | Time: 0.63s | Train E MAE: 2.80 meV/atom | Train F MAE: 42.65 meV/Å | Val E MAE: 2.61 meV/atom | Val F MAE: 37.71 meV/Å
Epoch 005 | Time: 0.63s | Train E MAE: 2.24 meV/atom | Train F MAE: 37.18 meV/Å | Val E MAE: 2.60 meV/atom | Val F MAE: 34.99 meV/Å
Epoch 006 | Time: 0.64s | Train E MAE: 4.82 meV/atom | Train F MAE: 34.78 meV/Å | Val E MAE: 1.23 meV/atom | Val F MAE: 33.17 meV/Å
Epoch 007 | Time: 0.64s | Train

Building graphs:   0%|          | 0/1000 [00:00<?, ?it/s]

  Done. Dataset ready (1000 graphs).
Epoch 000 | Time: 1.13s | Train E MAE: 19.01 meV/atom | Train F MAE: 196.98 meV/Å | Val E MAE: 6.55 meV/atom | Val F MAE: 178.68 meV/Å
Epoch 001 | Time: 0.95s | Train E MAE: 7.99 meV/atom | Train F MAE: 130.03 meV/Å | Val E MAE: 27.73 meV/atom | Val F MAE: 68.34 meV/Å
Epoch 002 | Time: 0.97s | Train E MAE: 10.73 meV/atom | Train F MAE: 50.17 meV/Å | Val E MAE: 5.86 meV/atom | Val F MAE: 37.11 meV/Å
Epoch 003 | Time: 0.94s | Train E MAE: 2.36 meV/atom | Train F MAE: 36.51 meV/Å | Val E MAE: 1.36 meV/atom | Val F MAE: 34.68 meV/Å
Epoch 004 | Time: 0.95s | Train E MAE: 0.66 meV/atom | Train F MAE: 34.11 meV/Å | Val E MAE: 0.59 meV/atom | Val F MAE: 32.13 meV/Å
Epoch 005 | Time: 0.93s | Train E MAE: 3.82 meV/atom | Train F MAE: 31.47 meV/Å | Val E MAE: 27.32 meV/atom | Val F MAE: 29.40 meV/Å
Epoch 006 | Time: 0.94s | Train E MAE: 9.10 meV/atom | Train F MAE: 28.91 meV/Å | Val E MAE: 2.60 meV/atom | Val F MAE: 27.28 meV/Å
Epoch 007 | Time: 0.97s | Train 

In [3]:
model = ACEWrapper(num_elements=120, num_radial=8, l_max=2, r_cut=5.0, hidden_dim=32)
# Check model size (number of parameters)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total Parameters: {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")

Total Parameters: 6,474
Trainable Parameters: 6,465
